# Datos Sintéticos — Caso 3

**Objetivo:** generar **100 casos sintéticos** estadísticamente similares a los 150 originales, para robustecer las reglas de decisión y probar el pipeline a escala.

**Estrategia:**

1. Aprender distribuciones marginales y correlaciones del dataset original con features.
2. Generar 100 filas sintéticas (copula gaussiana simplificada con numpy).
3. Generar `descripcion_reclamo` realista con **LLM vía OpenRouter** (`MODEL_SYNTHETIC_DATA` desde `.env`).
4. Validar similitud estadística (KS-test numéricas, chi-cuadrado categóricas).
5. Persistir en `data/synthetic_casos.parquet` y PostgreSQL (`es_sintetico=TRUE`).

## Outline

1. Setup y carga
2. Perfil estadístico del dataset original
3. Generación de variables numéricas
4. Generación de variables categóricas
5. Generación de descripciones con LLM
6. Validación estadística
7. Persistencia: parquet + PostgreSQL

## 1. Setup y carga

Cargamos el dataset **con features** (`casos_con_features.parquet`, output del step 02) para que los sintéticos respeten también las distribuciones de las variables derivadas.

In [1]:
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv

for candidata in [Path('.env'), Path('../.env')]:
    if candidata.exists():
        load_dotenv(candidata)
        break

DATA_DIR = Path('data') if Path('data').exists() else Path('../data')
df = pd.read_parquet(DATA_DIR / 'casos_con_features.parquet')

N_SINTETICOS = 100
rng = np.random.default_rng(seed=42)

print(f'Dataset base: {len(df)} casos, {len(df.columns)} columnas')
print(f'Modelo LLM: {os.getenv("MODEL_SYNTHETIC_DATA")}')

Dataset base: 150 casos, 32 columnas
Modelo LLM: meta-llama/llama-3.1-8b-instruct


## 2. Perfil estadístico del dataset original

Extraemos para cada variable numérica: media, desvío estándar, min, max. Y para las categóricas: distribución de frecuencias. Esto guía la generación.

In [2]:
NUM_BASE = [
    'antiguedad_usuario_dias', 'valor_orden_mxn', 'compensacion_solicitada_mxn',
    'num_compensaciones_90d', 'monto_compensado_90d_mxn',
    'tiempo_entrega_real_min', 'flags_fraude_previos',
]
CATS = ['ciudad', 'vertical', 'restaurante', 'entrega_confirmada_gps', 'motivo_reclamo']

perfil_num = df[NUM_BASE].agg(['mean', 'std', 'min', 'max']).T
print(perfil_num.round(2).to_string())

dist_cats = {c: df[c].value_counts(normalize=True) for c in CATS}

                               mean     std    min      max
antiguedad_usuario_dias      558.25  549.83   6.00  1796.00
valor_orden_mxn              329.15  159.41  98.76   699.80
compensacion_solicitada_mxn  260.06  131.83  69.54   692.52
num_compensaciones_90d         3.80    3.27   0.00    14.00
monto_compensado_90d_mxn     680.01  693.83   9.88  2763.77
tiempo_entrega_real_min       61.69   23.30  22.00   110.00
flags_fraude_previos           0.98    1.22   0.00     4.00


## 3. Generación de variables numéricas

Muestreamos con **distribución empírica + jitter gaussiano**: tomamos valores reales del dataset y les añadimos ruido pequeño proporcional a la desviación estándar. Esto preserva la forma de la distribución (incluyendo colas) mejor que asumir normalidad.

Respetamos las **restricciones lógicas**:

- `compensacion_solicitada ≤ valor_orden × 3` (techo razonable)
- `num_compensaciones_90d ≥ 0` y entero
- `monto_compensado_90d` coherente con `num_compensaciones_90d` (media ≈ monto/comp)
- Correlación `valor_orden ↔ compensacion` preservada (r ≈ 0.94)

In [3]:
def muestrear_empirico(serie: pd.Series, n: int, jitter: float = 0.05) -> np.ndarray:
    """Muestrea de la distribución empírica con ruido gaussiano.

    Args:
        serie: Columna original.
        n: Cantidad de muestras.
        jitter: Fracción de la std como ruido.

    Returns:
        Array con n valores sintéticos.
    """
    base = rng.choice(serie.dropna().values, size=n, replace=True)
    ruido = rng.normal(0, serie.std() * jitter, size=n)
    return base + ruido

syn = pd.DataFrame()
syn['valor_orden_mxn'] = np.clip(
    muestrear_empirico(df['valor_orden_mxn'], N_SINTETICOS),
    df['valor_orden_mxn'].min(), df['valor_orden_mxn'].max(),
)

# compensacion correlacionada con valor_orden (r≈0.94): ratio del dataset + ruido
ratios = (df['compensacion_solicitada_mxn'] / df['valor_orden_mxn'].clip(lower=1)).clip(0, 3)
syn['compensacion_solicitada_mxn'] = np.clip(
    syn['valor_orden_mxn'] * rng.choice(ratios.values, size=N_SINTETICOS, replace=True),
    df['compensacion_solicitada_mxn'].min(),
    df['compensacion_solicitada_mxn'].max(),
)

syn['antiguedad_usuario_dias'] = np.clip(
    muestrear_empirico(df['antiguedad_usuario_dias'], N_SINTETICOS),
    df['antiguedad_usuario_dias'].min(), df['antiguedad_usuario_dias'].max(),
).round().astype(int)

syn['num_compensaciones_90d'] = np.clip(
    muestrear_empirico(df['num_compensaciones_90d'], N_SINTETICOS),
    0, df['num_compensaciones_90d'].max(),
).round().astype(int)

# monto_compensado coherente: ≈ num_comps × compensación media individual
comp_media_individual = (
    df['monto_compensado_90d_mxn'] / df['num_compensaciones_90d'].clip(lower=1)
).clip(0, df['monto_compensado_90d_mxn'].max())
syn['monto_compensado_90d_mxn'] = np.clip(
    syn['num_compensaciones_90d']
    * rng.choice(comp_media_individual.values, size=N_SINTETICOS, replace=True),
    0, df['monto_compensado_90d_mxn'].max(),
)

syn['tiempo_entrega_real_min'] = np.clip(
    muestrear_empirico(df['tiempo_entrega_real_min'], N_SINTETICOS),
    df['tiempo_entrega_real_min'].min(), df['tiempo_entrega_real_min'].max(),
).round().astype(int)

syn['flags_fraude_previos'] = np.clip(
    muestrear_empirico(df['flags_fraude_previos'], N_SINTETICOS),
    0, df['flags_fraude_previos'].max(),
).round().astype(int)

print('Variables numéricas generadas:')
print(syn.describe().round(2).to_string())

Variables numéricas generadas:
       valor_orden_mxn  compensacion_solicitada_mxn  antiguedad_usuario_dias  num_compensaciones_90d  monto_compensado_90d_mxn  tiempo_entrega_real_min  flags_fraude_previos
count           100.00                       100.00                   100.00                  100.00                    100.00                   100.00                100.00
mean            343.88                       273.59                   540.16                    3.92                    729.55                    59.09                  0.65
std             162.96                       141.00                   495.62                    3.09                    745.26                    24.07                  1.05
min              98.76                        69.54                     6.00                    0.00                      0.00                    22.00                  0.00
25%             200.76                       160.17                   127.25                    2.0

## 4. Generación de variables categóricas

Muestreamos según la **distribución de frecuencias real** de cada columna. Así, si CDMX concentra el 18% de los casos originales, ~18% de los sintéticos serán de CDMX.

Para `usuario_id` generamos IDs sintéticos únicos (`USR-SYN-0001`…) para no colisionar con usuarios reales.

In [4]:
for col in CATS:
    dist = dist_cats[col]
    syn[col] = rng.choice(dist.index, size=N_SINTETICOS, p=dist.values)

syn['caso_id'] = [f'COMP-SYN-{i:04d}' for i in range(1, N_SINTETICOS + 1)]
syn['usuario_id'] = [f'USR-SYN-{i:04d}' for i in range(1, N_SINTETICOS + 1)]
syn['recomendacion_agente'] = 'PENDIENTE'
syn['es_sintetico'] = True

print('Distribuciones categóricas (sintético vs original):')
for col in ['ciudad', 'vertical', 'entrega_confirmada_gps']:
    comp = pd.DataFrame({
        'original': df[col].value_counts(normalize=True).round(3),
        'sintetico': syn[col].value_counts(normalize=True).round(3),
    })
    print(f'\n--- {col} ---')
    print(comp.to_string())

Distribuciones categóricas (sintético vs original):

--- ciudad ---
              original  sintetico
ciudad                           
Barranquilla     0.060       0.02
Bogotá           0.073       0.10
Buenos Aires     0.087       0.09
CDMX             0.180       0.18
Cali             0.033       0.08
Córdoba          0.047       0.04
Guadalajara      0.113       0.15
Lima             0.080       0.07
Medellín         0.040       0.06
Monterrey        0.033       0.01
Puebla           0.060       0.04
Querétaro        0.047       0.02
Santiago         0.053       0.05
São Paulo        0.033       0.03
Tijuana          0.060       0.06

--- vertical ---
          original  sintetico
vertical                     
Comida       0.527       0.51
Mercado      0.253       0.29
Farmacia     0.220       0.20

--- entrega_confirmada_gps ---
                        original  sintetico
entrega_confirmada_gps                     
NO confirmada              0.360       0.40
SÍ - confirmada       

## 5. Generación de descripciones con LLM

Usamos **OpenRouter** con el modelo `MODEL_SYNTHETIC_DATA` para redactar la `descripcion_reclamo` de cada caso sintético, coherente con sus variables.

El prompt inyecta el contexto del caso (vertical, ciudad, motivo, monto, GPS) para que el texto sea realista y consistente. Temperatura 0.8 para variedad sin perder coherencia.

**Fallback:** si el LLM falla para algún caso, se usa un template determinista.

In [5]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model=os.getenv('MODEL_SYNTHETIC_DATA'),
    api_key=os.getenv('OPENROUTER_API_KEY'),
    base_url='https://openrouter.ai/api/v1',
    temperature=0.8,
    max_tokens=100,
)

PROMPT_TEMPLATE = (
    'Escribe la descripción de un reclamo de un cliente de delivery en español, '
    'en primera persona, 1-2 oraciones naturales (sin saludos ni despedidas).\n'
    'Contexto del caso:\n'
    '- Motivo del reclamo: {motivo}\n'
    '- Vertical: {vertical} | Restaurante/tienda: {restaurante}\n'
    '- Ciudad: {ciudad}\n'
    '- Valor del pedido: ${valor:.0f} MXN | Compensación solicitada: ${comp:.0f} MXN\n'
    '- Entrega GPS: {gps} | Tiempo de entrega: {tiempo} min\n'
    'Responde SOLO con el texto del reclamo.'
)

def template_fallback(fila: pd.Series) -> str:
    """Genera una descripción determinista si el LLM falla.

    Args:
        fila: Fila del caso sintético.

    Returns:
        Texto del reclamo.
    """
    return (
        f'{fila["motivo_reclamo"]} en mi pedido de {fila["restaurante"]}. '
        f'Pagó ${fila["valor_orden_mxn"]:.0f} y solicito compensación de '
        f'${fila["compensacion_solicitada_mxn"]:.0f}.'
    )

print('LLM configurado. Generando descripciones...')

LLM configurado. Generando descripciones...


In [6]:
descripciones = []
errores_llm = 0

for i, fila in syn.iterrows():
    try:
        resp = llm.invoke(PROMPT_TEMPLATE.format(
            motivo=fila['motivo_reclamo'],
            vertical=fila['vertical'],
            restaurante=fila['restaurante'],
            ciudad=fila['ciudad'],
            valor=fila['valor_orden_mxn'],
            comp=fila['compensacion_solicitada_mxn'],
            gps=fila['entrega_confirmada_gps'],
            tiempo=fila['tiempo_entrega_real_min'],
        ))
        texto = resp.content.strip().strip('"')
        if len(texto) < 10:
            raise ValueError('respuesta muy corta')
        descripciones.append(texto)
    except Exception:
        descripciones.append(template_fallback(fila))
        errores_llm += 1
    if (i + 1) % 20 == 0:
        print(f'  {i + 1}/{N_SINTETICOS} generadas...')

syn['descripcion_reclamo'] = descripciones
print(f'\n[OK] {N_SINTETICOS} descripciones generadas ({errores_llm} con fallback)')
print('\nEjemplos:')
for t in descripciones[:3]:
    print(f'  - {t}')

  20/100 generadas...


  40/100 generadas...


  60/100 generadas...


  80/100 generadas...


  100/100 generadas...

[OK] 100 descripciones generadas (0 con fallback)

Ejemplos:
  - Estoy muy descontento, ya que mi pedido de Mariscos Premium nunca llegó y la única señal de la entrega fue que se perdió en mi ubicación, aunque el tiempo de entrega fue de aproximadamente 26 minutos. Me gustaría que me devuelvan el dinero que gasté en el pedido de $313 MXN, al menos el valor de las comidas que no recibí, que es de $225 MXN.
  - Soy un cliente que hizo un pedido en Sushi Fusión en São Paulo por $190 MXN, pero lamentablemente no recibí la orden en el tiempo estimado de 23 minutos y no se me confirmó la entrega a través del GPS. Estoy solicitando una compensación de $144 MXN.
  - Me encontré con un problema con mi pedido de Carnitas La Familia, que llegó hace 110 minutos pero desapareció la señal de GPS, me cobraron un monto incorrecto de $239 MXN, cuando me corresponden $221 MXN.


## 6. Validación estadística

- **KS-test** (numéricas): H₀ = misma distribución. Aceptamos p > 0.05.
- **Chi-cuadrado** (categóricas): frecuencias similares.
- **Correlación** `valor_orden ↔ compensacion`: debe quedar cerca de 0.94.

In [7]:
from scipy import stats

print('--- KS-test (p > 0.05 = distribución preservada) ---')
resultados_ks = {}
for col in NUM_BASE:
    stat, p = stats.ks_2samp(df[col].dropna(), syn[col].dropna())
    resultados_ks[col] = p
    marca = 'OK ' if p > 0.05 else 'BAJO'
    print(f'  [{marca}] {col}: p = {p:.4f}')

print('\n--- Chi-cuadrado (categóricas) ---')
for col in ['ciudad', 'vertical', 'entrega_confirmada_gps', 'motivo_reclamo']:
    obs = syn[col].value_counts().reindex(df[col].unique(), fill_value=1).values
    esp = df[col].value_counts(normalize=True).reindex(df[col].unique()).values * len(syn)
    stat, p = stats.chisquare(obs, esp)
    marca = 'OK ' if p > 0.05 else 'BAJO'
    print(f'  [{marca}] {col}: p = {p:.4f}')

r_syn = syn[['valor_orden_mxn', 'compensacion_solicitada_mxn']].corr().iloc[0, 1]
r_orig = df[['valor_orden_mxn', 'compensacion_solicitada_mxn']].corr().iloc[0, 1]
print(f'\nCorrelación valor↔comp: original={r_orig:.3f} sintético={r_syn:.3f}')

--- KS-test (p > 0.05 = distribución preservada) ---
  [OK ] antiguedad_usuario_dias: p = 0.8542
  [OK ] valor_orden_mxn: p = 0.6494
  [OK ] compensacion_solicitada_mxn: p = 0.4802
  [OK ] num_compensaciones_90d: p = 0.9170
  [OK ] monto_compensado_90d_mxn: p = 0.4038
  [OK ] tiempo_entrega_real_min: p = 0.8542
  [OK ] flags_fraude_previos: p = 0.1414

--- Chi-cuadrado (categóricas) ---
  [OK ] ciudad: p = 0.2857
  [OK ] vertical: p = 0.6821
  [OK ] entrega_confirmada_gps: p = 0.8551
  [OK ] motivo_reclamo: p = 0.3840

Correlación valor↔comp: original=0.941 sintético=0.931


## 7. Persistencia: parquet + PostgreSQL

Aplicamos el **mismo FeatureEngineer del step 02** a los sintéticos para que tengan las 16 features, y persistimos en parquet + DB (`es_sintetico=TRUE`).

Las features geográficas (`riesgo_ciudad`, `riesgo_vertical`) se calculan con las tasas del dataset ORIGINAL (los sintéticos heredan, no alteran la tasa).

In [8]:
# Recalcular features sobre los sintéticos con la misma lógica del step 02
P90_TIEMPO = df['tiempo_entrega_real_min'].quantile(0.90)
P95_COMP = df['compensacion_solicitada_mxn'].quantile(0.95)
P95_NCOMPS = df['num_compensaciones_90d'].quantile(0.95)
P90_NCOMPS = df['num_compensaciones_90d'].quantile(0.90)
PALABRAS = re.compile(
    r'alergi|intoxic|polic[ií]a|sangre|insult|denunci|abogado|demanda|hospital|veneno',
    re.IGNORECASE,
)

def aplicar_features(s: pd.DataFrame, ref: pd.DataFrame) -> pd.DataFrame:
    """Aplica el FeatureEngineer del step 02 a datos sintéticos.

    Args:
        s: DataFrame sintético con columnas base.
        ref: DataFrame original (para tasas geográficas y stats).
    """
    s = s.copy()
    s['comp_ratio'] = s['compensacion_solicitada_mxn'] / s['valor_orden_mxn'].clip(lower=1)
    s['burn_rate'] = s['monto_compensado_90d_mxn'] / s['antiguedad_usuario_dias'].clip(lower=1)
    s['freq_densidad'] = (
        s['num_compensaciones_90d'] / s['antiguedad_usuario_dias'].clip(upper=90).clip(lower=1)
    )
    s['flag_inconsistencia_gps'] = (
        (s['motivo_reclamo'] == 'Orden no llegó')
        & (s['entrega_confirmada_gps'].isin(['SÍ - confirmada', 'Parcial']))
    )
    s['flag_mentira_gps_alta'] = (
        s['motivo_reclamo'].isin(['Producto incorrecto', 'Producto incompleto'])
        & (s['entrega_confirmada_gps'] == 'SÍ - confirmada')
        & (s['compensacion_solicitada_mxn'] > P95_COMP)
    )
    s['flag_retraso_critico'] = s['tiempo_entrega_real_min'] > P90_TIEMPO
    s['flag_account_abuse'] = (
        (s['antiguedad_usuario_dias'] < 90) & (s['num_compensaciones_90d'] > P95_NCOMPS)
    )
    s['score_riesgo_previo'] = s['flags_fraude_previos'] * 2 + s['num_compensaciones_90d'] * 0.5
    texto = s['descripcion_reclamo'].fillna('')
    s['longitud_reclamo'] = texto.str.split().str.len()
    s['flag_palabras_criticas'] = texto.str.contains(PALABRAS)
    # Tasas del dataset ORIGINAL (heredadas, no recalculadas)
    tasa_ciudad = ref['ciudad'].value_counts(normalize=True)
    tasa_vertical = ref['vertical'].value_counts(normalize=True)
    s['riesgo_ciudad'] = s['ciudad'].map(tasa_ciudad).fillna(0)
    s['riesgo_vertical'] = s['vertical'].map(tasa_vertical).fillna(0)
    gps_ok = s['entrega_confirmada_gps'] == 'SÍ - confirmada'
    s['gps_paradoja_score'] = (
        gps_ok.astype(float) * 0.5
        + (s['num_compensaciones_90d'] > P90_NCOMPS).astype(float) * 0.3
        + (s['flags_fraude_previos'] > 0).astype(float) * 0.2
    )
    s['sospecha_nuevo_recurrente'] = (
        (s['antiguedad_usuario_dias'] < 90)
        & (s['num_compensaciones_90d'] >= 3)
        & (s['flags_fraude_previos'] >= 1)
    )
    media, std = ref['comp_ratio'].mean(), ref['comp_ratio'].std()
    s['ratio_deviation'] = (s['comp_ratio'] - media) / std if std > 0 else 0.0
    s['score_texto'] = np.nan
    return s

syn = aplicar_features(syn, df)
print(f'[OK] Features aplicadas: {len(syn.columns)} columnas totales')

[OK] Features aplicadas: 33 columnas totales


In [9]:
PARQUET_SYN = DATA_DIR / 'synthetic_casos.parquet'
syn.to_parquet(PARQUET_SYN, index=False)
print(f'[OK] Parquet: {PARQUET_SYN} ({len(syn)} filas)')

[OK] Parquet: ../data/synthetic_casos.parquet (100 filas)


In [10]:
import psycopg2
from psycopg2.extras import execute_values

DB_CONFIG = {
    'host': 'localhost', 'port': 5432, 'dbname': 'rappi_cases',
    'user': 'rappi', 'password': 'rappi_pass',
}

COLUMNAS_DB = [
    'caso_id', 'usuario_id', 'antiguedad_usuario_dias', 'ciudad', 'vertical',
    'restaurante', 'valor_orden_mxn', 'compensacion_solicitada_mxn',
    'num_compensaciones_90d', 'monto_compensado_90d_mxn', 'entrega_confirmada_gps',
    'tiempo_entrega_real_min', 'flags_fraude_previos', 'motivo_reclamo',
    'descripcion_reclamo', 'recomendacion_agente',
    'comp_ratio', 'burn_rate', 'freq_densidad', 'flag_inconsistencia_gps',
    'flag_mentira_gps_alta', 'flag_retraso_critico', 'flag_account_abuse',
    'score_riesgo_previo', 'longitud_reclamo', 'flag_palabras_criticas',
    'riesgo_ciudad', 'riesgo_vertical', 'gps_paradoja_score',
    'sospecha_nuevo_recurrente', 'ratio_deviation', 'score_texto', 'es_sintetico',
]

def a_tupla(fila: pd.Series) -> tuple:
    """Convierte una fila del DataFrame a tupla insertable en PostgreSQL.

    Args:
        fila: Fila con las columnas de COLUMNAS_DB.

    Returns:
        Tupla con tipos nativos de Python.
    """
    valores = []
    for col in COLUMNAS_DB:
        v = fila[col]
        if pd.isna(v):
            valores.append(None)
        elif isinstance(v, (bool, np.bool_)):
            valores.append(bool(v))
        elif isinstance(v, (np.integer,)):
            valores.append(int(v))
        elif isinstance(v, (np.floating,)):
            valores.append(float(v))
        else:
            valores.append(v)
    return tuple(valores)

conn = psycopg2.connect(**DB_CONFIG)
try:
    with conn.cursor() as cur:
        cur.execute("DELETE FROM casos WHERE es_sintetico = TRUE;")
        execute_values(
            cur,
            f"INSERT INTO casos ({', '.join(COLUMNAS_DB)}) VALUES %s",
            [a_tupla(f) for _, f in syn.iterrows()],
        )
    conn.commit()
finally:
    conn.close()

print(f'[OK] PostgreSQL: {len(syn)} casos sintéticos insertados')

[OK] PostgreSQL: 100 casos sintéticos insertados


In [11]:
# Verificación final
conn = psycopg2.connect(**DB_CONFIG)
check = pd.read_sql(
    'SELECT es_sintetico, COUNT(*) AS n FROM casos GROUP BY es_sintetico ORDER BY 1',
    conn,
)
conn.close()
print(check.to_string(index=False))
print('\n[OK] Step 05 completo: 100 sintéticos en parquet y PostgreSQL.')

 es_sintetico   n
        False 150
         True 100

[OK] Step 05 completo: 100 sintéticos en parquet y PostgreSQL.


/var/folders/fp/7jqy71qx7tg6j0vg_jwpvqrc0000gn/T/ipykernel_79013/2718301039.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  check = pd.read_sql(
